# 06 · Line profiles and optical depth

For every line j the gas at x absorbs the photon labelled by the observer's line-centred velocity v_† at the local
wavelength λ'_j = λ_j e^{v_†/c} a(x)/a_† D⁻¹(x) (notes `sec_flux.tex`, Eq. velocity_units_normalized), i.e. at the
frequency offset ν_j = c(1/λ' − 1/λ_j). The profile is a Voigt function φ = V(ν; σ, γ) with
σ = b/(√2 λ_j), b = √(2k_B T/m) + b_turb and γ = (A_21 + A_coll)/4π (Eq. line_profile), and

  τ_j(v_†) = C_j f_j Σ_i w_i n_com,i a_i⁻² φ_ij(v_†),  F = e^{−τ}  (Eq. tau_straight_los, discretised).

Runtime: ~40 s.

**Requires:** lines (01), skewers (03, a(x) from 02) and fields (05); the last section also simulation outputs (04).  
**Independent of:** the statistics (07).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import (use_style, ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, LABELS, FIGSIZES,
                                      add_panel_letters, plot_dag, plot_spectra, zoom_inset)
use_style()                                        # the dark, colour-blind-safe style of every figure (skewerjax.utils.plotting)
%matplotlib inline

In [ ]:
plot_dag(('tau',)); plt.show()

In [ ]:
from skewerjax.core import (LineBundle, LINES, voigt as V, line_wavelength_grid_A, frequency_offset_GHz, line_profiles,
                            compute_optical_depth, los_samples, resolution_report, wrap_to_ref_line, SkewerBundle, Fields,
                            ScalarField, VectorField, LogRamp, VelocityRamp, Constant, plane_points, analytic as an, phys)
lines = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"], ref_line_id="HI-LyA")
print(f"{len(LINES)} transitions in lines_table.dat")
for l, dv in zip(lines.lines, np.asarray(lines.delta_v)):
    print(f"  {l.id:11s} lambda0 = {l.lambda0_A:9.4f} A  f = {l.f_osc:.4f}  A21 = {l.A21_Hz:.3e} Hz  m = {l.mass_amu:7.3f} amu  delta_v = {dv:9.1f} km/s")

## 1 · The Voigt function

Three implementations of the same V(x; σ, γ): `"ExoJAX"` (default; Humlíček-type, jit/grad), `"convolution"`
(G * L by FFT on the grid; needs a grid that resolves γ) and `"scipy_wofz"` (NumPy reference, not traceable).

In [ ]:
x = jnp.linspace(-30, 30, 2401)
fig, (ax, axr) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, gridspec_kw={"height_ratios": [3, 1]}, layout="constrained")
for (s, g), c in zip([(1.0, 0.01), (1.0, 1.0), (0.3, 3.0)], [TOL["blue"], TOL["green"], TOL["red"]]):
    ref = V.voigt_scipy(np.asarray(x), np.asarray(s), np.asarray(g))
    ax.semilogy(x, ref, color=c, **MS["reference"])
    for m, sty in (("ExoJAX", MS["default"]), ("convolution", MS["matched"])):
        got = np.asarray(V.voigt_exojax(x, s, g) if m == "ExoJAX" else V.voigt_convolution(x, s, g, oversample=4))
        ax.semilogy(x, got, color=c, **sty, label=f"{m}, σ = {s}, γ = {g}" if s == 1.0 and g == 0.01 or m == "ExoJAX" else None)
        axr.plot(x, got / ref - 1, color=c, **sty)
ax.set(ylim=(1e-6, 2), ylabel="V(x; σ, γ)"); ax.legend(fontsize=13)
axr.set(xlabel="x", ylabel="/ wofz − 1", yscale="symlog", ylim=(-1, 1)); axr.yaxis.set_major_formatter(plt.matplotlib.ticker.FuncFormatter(lambda y, p: f"{y:g}"))
add_panel_letters([ax, axr]); fig.suptitle("Voigt: ExoJAX (solid) and FFT convolution (dashed) on scipy's wofz (light)", fontsize=22); plt.show()

ExoJAX agrees with wofz to < 10⁻⁶ in the core; the convolution is exact (10⁻¹⁴) when the fine grid resolves γ and degrades at the grid ends and for γ below the grid spacing (`scripts/validation/check_voigt.py`).

## 2 · Line profiles at points

`line_profiles(events, directions, v, a_†, lines, fields)` evaluates φ_ij for every line and point: a hotter gas
broadens the lines, a peculiar velocity along the photon shifts them (to larger v for gas moving towards the
observer... in the frame of the photon direction).

In [ ]:
pts = plane_points(1e-3, "y", 5e-4, 1, 5)[0]
events = jnp.concatenate([jnp.ones((5, 1)), pts], -1)
v = jnp.linspace(-400, 400, 801)
fig, axs = plt.subplots(1, 2, figsize=(20, 6), layout="constrained")
for ax, (Tlo, Thi, vlo, vhi, title) in zip(axs, ((1e3, 1e6, 0.0, 0.0, r"$T$ from $10^3$ to $10^6$ K"), (1e4, 1e4, -300.0, 300.0, r"$v_x$ from −300 to +300 km/s, $T = 10^4$ K"))):
    sp = {"temperature": ScalarField(LogRamp(Tlo, Thi, 1e-3, 3)), "velocity": VectorField(VelocityRamp(vlo, vhi, 1e-3)),
          "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0)), "number_density": ScalarField(Constant(1.0))}
    ev = events.at[:, 3].set(jnp.linspace(0, 1e-3, 5)) if Tlo != Thi else events.at[:, 1].set(jnp.linspace(0, 1e-3, 5))
    phi = np.asarray(line_profiles(ev, jnp.broadcast_to(jnp.array([1.0, 0, 0]), (5, 3)), v, 1.0, LineBundle(["HI-LyA"]), Fields({"HI": sp})))[0]
    for i in range(5):
        ax.semilogy(v, phi[i], color=list(TOL.values())[i])
    ax.set(xlabel=r"$v_\dagger$ [km/s]", ylabel=r"$\phi$ [ns]", ylim=(1e-8, None), title=title)
add_panel_letters(axs); fig.suptitle(r"Ly$\alpha$ profiles: temperature sets the width, the line-of-sight velocity the centre", fontsize=22); plt.show()

The detuning uses the exact exponential mapping by default; `detuning="linear"` is fake_spectra's first-order version (matched mode, notebook 08). `velocity_period=P` folds the detuning into [−P/2, P/2) for periodic boxes.

## 3 · Optical depth: checks against closed forms

Static uniform gas (a = 1, no velocities) has closed forms (notes `app_analytical_solutions.tex`): a Lorentzian for
T = 0 and a Gaussian for A_21 = 0 (the `null…` lines of the table). Lines of one species share the fields.

In [ ]:
Lk = 1e-3
def fields(T, n):
    sp = lambda: {"temperature": ScalarField(LogRamp(*T, Lk, 3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, Lk)),
                  "number_density": ScalarField(LogRamp(*n, Lk, 2)), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
    return Fields({"HI": sp(), "SiIII": sp()})
null = LineBundle(["nullHI-LyA", "nullHI-LyB", "nullSiIII1206"])
pts = plane_points(Lk, "y", Lk / 2, 4, 32)                                       # 4 skewers, temperature rising with z
sb = SkewerBundle(spatial_points=pts, manual_scale_factors=jnp.ones_like(pts[None, ..., 0]))
v = jnp.linspace(-3000.0, 3000.0, 1024)
F_th = fields((1e4, 1e8), (1e-7, 1e-7))
tau = compute_optical_depth(sb, v, null, F_th, a_dagger=1.0)
T_sk = F_th["HI"]["temperature"](sb.quadri_vectors[0])[:, 0]
wl = line_wavelength_grid_A(v, null, 1.0, 1.0)
ref = jnp.stack([an.tau_static_thermal(wl[j], null.lambda0_A[j:j+1], null.mass_amu[j:j+1], null.f_osc[j:j+1], sb.a0, Lk * jnp.ones(4),
                                       1e-7 * jnp.ones(4), T_sk)[0] for j in range(3)])
F, Fa = np.exp(-np.asarray(tau)), np.exp(-np.asarray(ref))
fig, axs = plt.subplots(2, 3, figsize=(22, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
for j, l in enumerate(lines.lines):
    for k in range(4):
        c = list(TOL.values())[k]
        axs[0, j].plot(v, Fa[j, k], color=c, **MS["reference"]); axs[0, j].plot(v, F[j, k], color=c, **MS["default"], label=f"T = {float(T_sk[k]):.0e} K" if j == 0 else None)
        axs[1, j].plot(v, F[j, k] - Fa[j, k], color=c)
    axs[0, j].set_title(l.name); axs[1, j].set_xlabel(r"$v_\dagger$ [km/s]")
axs[0, 0].set_ylabel("F"); axs[1, 0].set_ylabel(r"$F - F_{\rm analytic}$"); axs[0, 0].legend(fontsize=12)
add_panel_letters(axs); fig.suptitle("Static thermal lines: SkewerJAX (solid) on the analytic Gaussians (light); residuals < 10⁻⁸", fontsize=22); plt.show()

Radiation-dominated expansion and the convergence with the sampling are checked in `scripts/validation/check_optical_depth_analytic.py`.

## 4 · Line-centred frames and the global frame

Every line j is integrated on the same grid of **line-centred velocities** v_† (its own frame): the gas at a, moving
with v_∥ along the photon, absorbs the photon labelled v_† at λ'_j = λ_j e^{v_†/c} (a/a_†) D⁻¹ (notes
`app_velocity_units.tex`). Its line centre sits at the same v_c in every frame (`core.spectral.line_centre_velocity_kms`
L86):

$$v_c = c\Big[\ln\frac{a_\dagger}{a} + \tfrac12\ln\frac{1-\beta}{1+\beta}\Big],\qquad \beta = v_\parallel/c .$$

The observer sees one **global** axis. In observed wavelength, λ_obs = λ_j (a_obs/a_†) e^{v_†/c}
(`spectral.line_wavelength_grid_A` L131); on the Lyα frame, the same pixel of line j sits at v_Lyα = v_† − Δv_j with
Δv_j = c ln(λ_Lyα/λ_j) (`LineBundle.delta_v`, `optical_depth.wrap_to_ref_line` L351).

**One absorber, three lines.** A single cold point (T → 0, so φ is the Lorentzian of A21) at a = a_† e^{−v_c/c},
v_c = 150 km/s, seen by an observer with a_† = 1/3. The analytic profile is the Lorentzian γ/π/(ν² + γ²) with the exact
detuning ν = (c/λ_j)[e^{−(v_†−v_c)/c} − 1] (notes `sec_flux.tex`, Eq. line_profile), written out here in NumPy.

In [ ]:
frames = LineBundle(["HI-LyA", "HI-LyB", "SiIII-1206"], ref_line_id="HI-LyA")
a_dag, v_c = 1.0 / 3.0, 150.0
a_abs = a_dag * np.exp(-v_c / phys.c_kms)
T_tiny = float(np.finfo(np.float32).tiny)                    # T -> 0: the Gaussian width vanishes
cold = lambda: {"temperature": ScalarField(Constant(T_tiny)), "velocity": VectorField(VelocityRamp(0.0, 0.0, 1.0)),
                "number_density": ScalarField(Constant(1.0)), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
vv = jnp.linspace(-150.0, 450.0, 3001)
phi = np.asarray(line_profiles(jnp.array([[a_abs, 0.0, 0.0, 0.0]]), jnp.array([[1.0, 0.0, 0.0]]), vv, a_dag, frames,
                               Fields({"HI": cold(), "SiIII": cold()})))[:, 0]
lam0, gam = np.asarray(frames.lambda0_A)[:, None], np.asarray(frames.A21_GHz)[:, None] / (4 * np.pi)
nu = phys.c * 10.0 / lam0 * np.expm1(-(np.asarray(vv)[None] - v_c) / phys.c_kms)          # exact detuning [GHz]
phi_an = gam / np.pi / (nu ** 2 + gam ** 2)                                               # Lorentzian [ns]
from skewerjax.utils.plotting import plot_line_frames, plot_flux_vs_analytic
fig, _ = plot_line_frames(np.asarray(vv), phi, phi_an, frames, v_c, a_dagger=a_dag, a_obs=1.0)
fig.suptitle("One absorber: the same line-centre velocity in every own frame (top), different places on the global axis (bottom)", fontsize=21)
plt.show()
print(f"{'line':11s} {'Delta v_j':>10s} {'lambda0 a_obs/a_dag':>20s} {'lambda_c (absorber)':>20s} {'v on the Lya frame':>19s} {'max |phi/phi_an-1|':>19s}")
for j, l in enumerate(frames.lines):
    dvj = float(frames.delta_v[j])
    core_ = np.abs(np.asarray(vv) - v_c) < 50
    print(f"{l.id:11s} {dvj:10.1f} {l.lambda0_A * 3:20.3f} {l.lambda0_A * 3 * np.exp(v_c / phys.c_kms):20.3f} {v_c - dvj:19.1f} "
          f"{np.max(np.abs(phi[j][core_] / phi_an[j][core_] - 1)):19.1e}")

*Reading the figure.* In its own frame every line peaks at v_c = 150 km/s (dashed), 150 km/s from its rest-frame
centre (dash-dot), with the Lorentzian peak 4/A21 (×). On the observed-wavelength axis the same absorber gives three
lines at λ_j (a_obs/a_†) e^{v_c/c}; on the Lyα frame they sit at v_c − Δv_j (table). The residual panels show the
agreement with the analytic Lorentzian (the band is ±10⁻⁶; the printed maximum is over the ±50 km/s around the centre).

**Many absorbers: the optical depth against the closed form.** Radiation-dominated expansion with uniform gas
(n0 = 5×10⁻¹¹ cm⁻³, A21 = 0, so the closed form is a difference of error functions; notes
`app_analytical_solutions.tex`, `core.analytic.tau_radiation_thermal`), on two 5 cMpc skewers: one with the initial
conditions at the absorption end (a0 = 1), one at the emission end, at two temperatures (after the pre-refactor
`6_optical_depth`, cells 28–32; the convergence with the sampling is `scripts/validation/check_optical_depth_analytic.py`).

In [ ]:
from skewerjax.core import Cosmology
rad = Cosmology(H0=phys.H0, Omega_r_0=1.0, Omega_m_0=0.0, Omega_k_0=0.0, Omega_lambda_0=0.0)
L5, n_pt, T0, n0 = 5.0, 336, jnp.array([5e4, 5e6]), 5e-11
sbr = SkewerBundle(spatial_points=plane_points(L5, "y", L5 / 2, 2, n_pt), a0=[1.0, 0.99999976], x0=[0.0, 0.0], ICs_defined_at_index=[-1, 0],
                   ICs_at_emission_or_absorption=["absorption", "emission"], cosmologies=rad)
sp_r = lambda: {"temperature": ScalarField(LogRamp(5e4, 5e6, L5, 3)), "velocity": VectorField(VelocityRamp(0.0, 0.0, L5)),
                "number_density": ScalarField(LogRamp(n0, n0, L5, 2)), "turbulence": ScalarField(Constant(0.0)), "collisional": ScalarField(Constant(0.0))}
v_r = jnp.linspace(-4000.0, 4000.0, 2048)
tau_r = compute_optical_depth(sbr, v_r, null, Fields({"HI": sp_r(), "SiIII": sp_r()}), a_dagger=jnp.repeat(jnp.asarray(sbr.a0)[:, None], n_pt, 1),
                              resolution_tolerance=None)
ref_r = jnp.stack([an.tau_radiation_thermal(line_wavelength_grid_A(v_r, null, 1.0, 1.0)[j], null.lambda0_A[j:j + 1], null.f_osc[j:j + 1],
                                            null.mass_amu[j:j + 1], T0, sbr.a0, sbr.x0, jnp.array([-L5, 0.0]), jnp.array([0.0, L5]),
                                            jnp.full(2, n0))[0] for j in range(3)])
a_far = np.asarray(sbr.scale_factors[0])[np.arange(2), np.where(np.asarray(sbr.ICs_defined_at_index) == 0, -1, 0)]
lam_obs = np.stack([np.stack([np.asarray(line_wavelength_grid_A(v_r, null, a0, a0))[j] for a0 in np.asarray(sbr.a0)]) for j in range(3)])
lam_c = np.asarray(null.lambda0_A)[:, None] * np.asarray(sbr.a0)[None] / a_far[None]
fig, _ = plot_flux_vs_analytic(np.asarray(v_r), np.exp(-np.asarray(tau_r)), np.exp(-np.asarray(ref_r)), null, lam_obs, lam_c,
                               skewer_labels=[f"ICs at absorption, a0 = 1, T = 5e4 K", "ICs at emission, a0 = 0.99999976, T = 5e6 K"])
fig.suptitle("Radiation domination, thermal lines: numerical (solid) on the analytic F (light); zooms ±80 km/s and residuals", fontsize=21)
plt.show()
print("max |F - F_analytic| per line:", np.round(np.abs(np.exp(-np.asarray(tau_r)) - np.exp(-np.asarray(ref_r))).max(axis=(1, 2)), 5),
      f"({n_pt} points; second order in the spacing, check_optical_depth_analytic.py)")

*Reading the figure.* (a) On the observed-wavelength axis each line absorbs between its rest wavelength λ0
(dash-dot, the end of the path at a0) and λ0 a0/a_far (dashed), where the light left the other end of the box: the
Hubble flow across 5 cMpc spreads the absorption over a few hundred km/s, to the red or to the blue of λ0 depending
on where the initial conditions sit. (b–d) Around each line's v_† = 0 the numerical F lies on the closed form, and
(e–g) the residuals stay inside the ±10⁻² band; they fall as the square of the point spacing.

## 5 · Options and the layers underneath

`compute_optical_depth(bundle, v, lines, fields, *, a_dagger, ii_cosmology, line_elements, voigt, detuning,
velocity_period, resolution_tolerance, v_chunk, batch_size)` is three pure layers:
`los_samples` → `LoSSamples` (every field at every point, a pytree; the boundary used by `skewerjax.simulations`),
`optical_depth_from_samples` (jit; `lax.map` over skewers, `batch_size` of them vmapped at once, `v_chunk` velocities
per call) and `optical_depth_kernel` (one skewer). `resolution_report` says whether the points resolve the lines
(notes `app_numerical_resolution.tex`).

In [ ]:
F_moving = Fields({sp: {"temperature": ScalarField(LogRamp(2e4, 2e4, Lk)), "velocity": VectorField(VelocityRamp(-300.0, 300.0, Lk)),
                         "number_density": ScalarField(LogRamp(1e-6, 1e-6, Lk)), "turbulence": ScalarField(Constant(0.0)),
                         "collisional": ScalarField(Constant(0.0))} for sp in ("HI", "SiIII")})
pts_x = jnp.stack([jnp.linspace(0, Lk, 1024), jnp.full(1024, Lk / 2), jnp.full(1024, Lk / 2)], -1)[None]   # along x: v_x ramps
for n_pt in (1024, 64):
    sbx = SkewerBundle(spatial_points=pts_x[:, ::1024 // n_pt], manual_scale_factors=jnp.ones((1, 1, n_pt)))
    rep = resolution_report(los_samples(sbx, F_moving, lines, a_dagger=1.0), lines, 0.25)
    print(f"{n_pt:4d} points: max |dv_c| / width per line = {np.round(rep['max_ratio'], 2)} -> {'resolved' if rep['ok'] else 'UNDER-RESOLVED (compute_optical_depth warns)'}")
sbx = SkewerBundle(spatial_points=pts_x, manual_scale_factors=jnp.ones((1, 1, 1024)))
v = jnp.linspace(-600, 2900, 1400)
lya_si = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
tau = compute_optical_depth(sbx, v, lya_si, F_moving, a_dagger=1.0)
tau_ref = wrap_to_ref_line(tau, v, lya_si)
fig, axs = plt.subplots(1, 2, figsize=(20, 6), layout="constrained", sharey=True)
for j, l in enumerate(lya_si.lines):
    axs[0].plot(v, np.exp(-tau[j, 0]), color=EC[l.id], label=l.id); axs[1].plot(v, np.exp(-tau_ref[j, 0]), color=EC[l.id])
axs[0].set(xlabel=r"$v_\dagger$ [km/s], each line in its own frame", ylabel="F", title="the same gas, two lines")
axs[1].set(xlabel=r"$v$ [km/s] on the Ly$\alpha$ frame (wrap_to_ref_line)", title=r"Si III moves by $\Delta v$ = 2270 km/s")
axs[0].legend(); add_panel_letters(axs); plt.show()

## 6 · Option sweep: detuning and precision

The same moving gas with three settings of the τ kernel:
- the exact detuning (default);
- fake_spectra's linear detuning (`detuning="linear"`);
- the opt-in single-precision kernel (`dtype="float32"`; the result is returned in float64).

The panels below show the differences from the default.

In [ ]:
t_lin = compute_optical_depth(sbx, v, lya_si, F_moving, a_dagger=1.0, detuning="linear")
t_f32 = compute_optical_depth(sbx, v, lya_si, F_moving, a_dagger=1.0, dtype="float32")
fig, axs = plt.subplots(2, 1, figsize=FIGSIZES["tall"], sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
plot_spectra(axs[0], v, np.exp(-np.asarray(tau)[:, 0]), [l.id for l in lya_si.lines], total=False)
for j, l in enumerate(lya_si.lines):
    axs[1].plot(v, np.exp(-t_lin[j, 0]) - np.exp(-tau[j, 0]), color=EC[l.id], **MS["matched"], label=f"{l.id}: linear − exact")
    axs[1].plot(v, np.exp(-t_f32[j, 0]) - np.exp(-tau[j, 0]), color=EC[l.id], **MS["alternative"], label=f"{l.id}: float32 − float64")
axs[1].set(xlabel=LABELS["v_dagger"], ylabel=LABELS["dF"], yscale="symlog"); axs[1].set_yscale("symlog", linthresh=1e-8)
axs[0].legend(); axs[1].legend(fontsize=11, ncol=2)
zoom_inset(axs[0], (-150, 150), (-0.02, 1.02), bounds=(0.62, 0.12, 0.35, 0.5))
add_panel_letters(axs); fig.suptitle("The default (a), and what the detuning and precision options change (b)", fontsize=21); plt.show()
print(f"max |dF|: linear detuning {np.abs(np.exp(-t_lin) - np.exp(-tau)).max():.1e}, float32 {np.abs(np.exp(-t_f32) - np.exp(-tau)).max():.1e}")

## 7 · JAX: jit, vmap, grad

Every function above is pure; the containers are pytrees. For example, the derivative of the mean flux with respect
to the density of the gas:

In [ ]:
def mean_flux(n0):
    f = Fields({sp: {"temperature": ScalarField(LogRamp(2e4, 2e4, Lk)), "velocity": VectorField(VelocityRamp(-300.0, 300.0, Lk)),
                     "number_density": ScalarField(Constant(n0)), "turbulence": ScalarField(Constant(0.0)),
                     "collisional": ScalarField(Constant(0.0))} for sp in ("HI", "SiIII")})
    return jnp.exp(-compute_optical_depth(sbx, v, lya_si, f, a_dagger=1.0, resolution_tolerance=None)[0]).mean()
n0 = 1e-6
print(f"<F> = {float(mean_flux(n0)):.4f}, d<F>/d ln n = {float(jax.grad(mean_flux)(n0) * n0):.4f}, "
      f"finite difference {float((mean_flux(n0 * 1.001) - mean_flux(n0 * 0.999)) / 0.002):.4f}")

## 8 · τ from simulation outputs

The same integrator on the gas of notebook 04. `SimulationSkewers.optical_depth(lines)` integrates every line in its
own frame over one period of the box (1 km/s pixels by default), from the deposited pieces of every dual segment;
`fields(species)` gives the same gas as a `core.Fields` at the points.

In [ ]:
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
from skewerjax.core import Cosmology
from skewerjax.simulations import ArepoHDF5Reader, SimulationSkewers, IntegrationOptions, HI
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
cosmo = Cosmology(H0=100 * H.h * 1e6 * phys.Gyr_to_s / phys.Gpc_to_m, Omega_r_0=0.0, Omega_m_0=H.omega_m, Omega_k_0=0.0, Omega_lambda_0=H.omega_l)
L = H.box_cMpc
x = np.linspace(0.0, L, int(round(H.vmax_kms / 0.25)) + 1)
points = np.stack([np.stack([x, np.full_like(x, y), np.full_like(x, z)], -1) for y, z in [(10.0, 30.0), (25.0, 5.0), (40.0, 45.0)]])
bundle = SkewerBundle(spatial_points=jnp.asarray(points), a0=H.a, ICs_defined_at_index=0, ICs_at_emission_or_absorption="absorption", cosmologies=cosmo)
sk = SimulationSkewers.from_bundle(reader, bundle)
print(f"TNG50-4 snapshot 25: the three skewers of notebook 04, {len(sk.cells):,} cells selected in {sk.selection_s:.1f} s")

In [ ]:
lines = LineBundle(["HI-LyA", "SiIII-1206"], ref_line_id="HI-LyA")
velocities, tau = sk.optical_depth(lines)
flux = np.exp(-tau)
fields_sim = sk.fields(lines.species_keys)
sampled = fields_sim.sample(bundle.quadri_vectors[0], bundle.los_directions)
a_pts = np.asarray(bundle.quadri_vectors[0][..., 0])
v_pts = phys.c_kms * np.log(a_pts[:, :1] / a_pts)
fig, axs = plt.subplots(3, 3, figsize=(22, 12), sharex=True, sharey="row", layout="constrained")
for k in range(3):
    for j, l in enumerate(lines.lines):
        sty = MS["default"] if j == 0 else MS["matched"]
        axs[0, k].semilogy(v_pts[k], np.asarray(sampled[l.species]["number_density"])[k] / H.a ** 3, color=EC[l.species], **sty)
        axs[2, k].plot(velocities, flux[j, k], color=EC[l.species], **sty, label=l.id)
    axs[1, k].semilogy(v_pts[k], np.asarray(sampled["HI"]["temperature"])[k], color=EC["temperature"])
    axs[0, k].set_title(f"skewer {k}: y = {points[k, 0, 1]:.0f}, z = {points[k, 0, 2]:.0f} cMpc", fontsize=18); axs[2, k].set_xlabel("v [km/s]")
axs[0, 0].set_ylabel(r"$n$ [cm$^{-3}$]"); axs[1, 0].set_ylabel("T [K]"); axs[2, 0].set_ylabel("F"); axs[2, 0].legend()
add_panel_letters(axs, size=16); fig.suptitle("Three skewers through TNG50-4: the gas (top) and the flux it causes (bottom)", fontsize=22); plt.show()
print(f"<F>: Lya {flux[0].mean():.3f}, Si III {flux[1].mean():.4f}")

### Precision, zoom-in and devices

The τ of the three skewers again, with the opt-in float32 kernel (`IntegrationOptions(dtype="float32")`). The
float64 default is the reference. The inset zooms into one absorber. The same calls take `devices=jax.devices()`
to shard the work over several devices (notebook 08 and `docs/usage.md`).

In [ ]:
import time
t0 = time.perf_counter(); _, tau64 = sk.optical_depth(lines); t64 = time.perf_counter() - t0
t0 = time.perf_counter(); _, tau32 = sk.optical_depth(lines, integ=IntegrationOptions(dtype="float32")); t32 = time.perf_counter() - t0
fig, axs = plt.subplots(2, 1, figsize=FIGSIZES["tall"], sharex=True, gridspec_kw={"height_ratios": [2, 1]}, layout="constrained")
plot_spectra(axs[0], velocities, np.exp(-tau64[:, 0]), [l.id for l in lines.lines], total=False)
for j, l in enumerate(lines.lines):
    axs[1].plot(velocities, np.exp(-tau32[j, 0]) - np.exp(-tau64[j, 0]), color=EC[l.id], lw=1.2, label=l.id)
k0 = int(np.argmin(np.exp(-tau64[0, 0])))
zoom_inset(axs[0], (velocities[k0] - 60, velocities[k0] + 60), (-0.02, 1.02), bounds=(0.66, 0.1, 0.32, 0.5))
axs[1].set(xlabel=LABELS["v_dagger"], ylabel=r"$F_{32} - F_{64}$"); axs[0].legend(); axs[1].legend()
add_panel_letters(axs); fig.suptitle(f"float32 kernel: max |ΔF| = {np.abs(np.exp(-tau32) - np.exp(-tau64)).max():.1e}, "
                                     f"{t64:.1f} s → {t32:.1f} s (first calls include compilation)", fontsize=20); plt.show()

## Summary

- The Voigt profile (ExoJAX) agrees with SciPy's `wofz` to < 1e-6 in the line core. τ matches the static closed
  forms to < 1e-8 in F.
- Temperature sets the width and the line-of-sight velocity the centre. Every line is computed in its own frame.
- The options change the result as documented: the linear detuning differs from the exact one only far from the line
  centre, and the opt-in float32 kernel changes F by < 1e-6 here.
- On simulation outputs the flux follows the gas along each skewer; the opt-in float32 kernel changes F by
  ~1e-3 at most (2–3× faster on the CPU), the float64 default is the reference.
- Checks: `scripts/validation/check_voigt.py`, `check_optical_depth_analytic.py`, `check_line_frames.py`, `check_tng_optical_depth.py`.